# 🔵 Modelo: K-Nearest Neighbors (KNN)
**Dataset:** Sonar – Classificação de Minas vs Rochas  
**Algoritmo:** KNeighborsClassifier (scikit-learn)  
**Parâmetros:** k=5 | métrica=Minkowski (p=2, Euclidiana)  
**Split:** 75% treino / 25% teste | random_state=42  
**Normalização:** StandardScaler


## 1. Importações e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score
)
# Bibliotecas de monitoramento de performance computacional
import time
import tracemalloc
import platform
import psutil
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120


## 2. Carregamento dos Dados

In [ ]:
PATH = r"C:\Users\Usuário\Documents\paradigmas-programacao\dataset_1_matriz\sonar_data_matriz.csv"

dataset = pd.read_csv(PATH, header=None)
print("Shape:", dataset.shape)
print("\nDistribuição de classes:")
print(dataset.iloc[:, -1].value_counts())
dataset.head()


## 3. Pré-processamento e Normalização

In [ ]:
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values
y = np.where(y == 'M', 1, 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# KNN é sensível à escala — normalização obrigatória
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Treino : {X_train_s.shape[0]} amostras  |  Teste: {X_test_s.shape[0]} amostras")
print("Normalização: StandardScaler aplicado ✔")


## 4. Escolha Ótima de K (Elbow)

In [ ]:
k_range = range(1, 21)
acc_list, f1_list = [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    knn.fit(X_train_s, y_train)
    yp = knn.predict(X_test_s)
    acc_list.append(accuracy_score(y_test, yp))
    f1_list.append(f1_score(y_test, yp))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_range, acc_list, 'o-', color='#4CAF50', label='Acurácia')
ax.plot(k_range, f1_list,  's--', color='#FF9800', label='F1-Score')
ax.axvline(5, color='red', ls=':', lw=1.5, label='k=5 (usado)')
ax.set_xlabel('Número de Vizinhos (k)'); ax.set_ylabel('Score')
ax.set_title('Impacto de k no Desempenho – KNN', fontweight='bold')
ax.legend(); ax.set_xticks(list(k_range))
plt.tight_layout()
plt.savefig('../images/knn_elbow.png', bbox_inches='tight')
plt.show()


## 5. Treinamento com k=5

In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
modelo_knn.fit(X_train_s, y_train)
y_pred  = modelo_knn.predict(X_test_s)
y_proba = modelo_knn.predict_proba(X_test_s)[:, 1]

print("Modelo KNN (k=5) treinado ✔")


## 6. Métricas de Desempenho

In [ ]:
acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)
cv_scores = cross_val_score(modelo_knn, scaler.fit_transform(X), y, cv=5, scoring='accuracy')

metricas = {
    "Acurácia"         : acc,
    "Precisão (Mina)"  : precision,
    "Recall (Mina)"    : recall,
    "F1-Score (Mina)"  : f1,
    "ROC-AUC"          : roc_auc,
    "CV Acurácia (5x)" : cv_scores.mean(),
}

df_metricas = pd.DataFrame(metricas.items(), columns=["Métrica", "Valor"])
df_metricas["Valor"] = df_metricas["Valor"].map(lambda v: f"{v:.4f}")
print("=== MÉTRICAS – KNN ===")
display(df_metricas)


## 7. Performance Computacional

> Métricas de uso de CPU, memória RAM e tempo de execução coletadas durante  
> o treinamento e a inferência do modelo.


In [ ]:
# ── Informações do ambiente ──────────────────────────────────────────────────
print("=== AMBIENTE DE EXECUÇÃO ===")
print(f"  SO          : {platform.system()} {platform.release()}")
print(f"  Processador : {platform.processor() or platform.machine()}")
print(f"  Núcleos CPU : {psutil.cpu_count(logical=False)} físicos / {psutil.cpu_count()} lógicos")
print(f"  RAM total   : {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"  Python      : {platform.python_version()}")

# ── CPU e RAM antes do treino ────────────────────────────────────────────────
cpu_antes = psutil.cpu_percent(interval=0.5)
proc      = psutil.Process(os.getpid())
ram_antes = proc.memory_info().rss / 1e6

# ── Treino com medição de tempo e memória ────────────────────────────────────
tracemalloc.start()
t0_treino = time.perf_counter()

_modelo_perf = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
_modelo_perf.fit(X_train_s, y_train)

t1_treino = time.perf_counter()
mem_pico_treino, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

tempo_treino       = t1_treino - t0_treino
mem_pico_treino_mb = mem_pico_treino / 1e6

# ── Inferência com medição de tempo ─────────────────────────────────────────
t0_inf = time.perf_counter()
_ = _modelo_perf.predict(X_test_s)
t1_inf = time.perf_counter()
tempo_inferencia  = t1_inf - t0_inf
tempo_por_amostra = tempo_inferencia / len(X_test) * 1000

# ── CPU e RAM depois ─────────────────────────────────────────────────────────
cpu_depois = psutil.cpu_percent(interval=0.5)
ram_depois = proc.memory_info().rss / 1e6

# ── Tabela de resultados ─────────────────────────────────────────────────────
df_perf = pd.DataFrame({
    "Métrica": [
        "Tempo de Treino (s)",
        "Tempo de Inferência (s)",
        "Tempo por Amostra (ms)",
        "Pico de Memória — Treino (MB)",
        "RAM do Processo — Antes (MB)",
        "RAM do Processo — Depois (MB)",
        "Delta RAM (MB)",
        "CPU antes do treino (%)",
        "CPU depois do treino (%)",
        "Amostras de Treino",
        "Amostras de Teste",
        "Núcleos lógicos disponíveis",
    ],
    "Valor": [
        f"{tempo_treino:.6f}",
        f"{tempo_inferencia:.6f}",
        f"{tempo_por_amostra:.4f}",
        f"{mem_pico_treino_mb:.4f}",
        f"{ram_antes:.2f}",
        f"{ram_depois:.2f}",
        f"{ram_depois - ram_antes:.2f}",
        f"{cpu_antes:.1f}",
        f"{cpu_depois:.1f}",
        str(len(X_train)),
        str(len(X_test)),
        str(psutil.cpu_count()),
    ]
})

print("\n=== PERFORMANCE COMPUTACIONAL — KNN ===")
display(df_perf)


In [ ]:
# ── Visualização das métricas de performance ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

labels_t = ["Treino", "Inferência"]
vals_t   = [tempo_treino, tempo_inferencia]
bars = axes[0].bar(labels_t, vals_t, color=["#4CAF50", "#FF9800"], edgecolor="white", width=0.5)
for b, v in zip(bars, vals_t):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + max(vals_t)*0.01,
                 f"{v:.4f}s", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_title("Tempo de Execução (s)", fontweight="bold")
axes[0].set_ylabel("Segundos")

labels_m = ["RAM antes", "RAM depois", "Pico treino"]
vals_m   = [ram_antes, ram_depois, mem_pico_treino_mb]
bars = axes[1].bar(labels_m, vals_m, color=["#4CAF50", "#E57373", "#FFB74D"], edgecolor="white", width=0.5)
for b, v in zip(bars, vals_m):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + max(vals_m)*0.01,
                 f"{v:.1f} MB", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].set_title("Uso de Memória RAM (MB)", fontweight="bold")
axes[1].set_ylabel("MB")

labels_c = ["CPU antes (%)", "CPU depois (%)"]
vals_c   = [cpu_antes, cpu_depois]
bars = axes[2].bar(labels_c, vals_c, color=["#4CAF50", "#78909C"], edgecolor="white", width=0.5)
for b, v in zip(bars, vals_c):
    axes[2].text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                 f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[2].set_ylim(0, 110)
axes[2].set_title("Utilização de CPU (%)", fontweight="bold")
axes[2].set_ylabel("%")

fig.suptitle("Performance Computacional — KNN (k=5)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 9. Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Rocha (0)', 'Mina (1)'],
            yticklabels=['Rocha (0)', 'Mina (1)'],
            ax=axes[0], linewidths=.5)
axes[0].set_title('Matriz de Confusão – KNN', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predito')

labels = ['VN\n(Rocha correta)', 'FP\n(Falso alarme)', 'FN\n(Mina perdida)', 'VP\n(Mina correta)']
values = [cm[0,0], cm[0,1], cm[1,0], cm[1,1]]
colors = ['#4CAF50', '#FF5252', '#FF9800', '#2196F3']
bars = axes[1].bar(labels, values, color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Componentes da Matriz de Confusão', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.savefig('../images/knn_matriz_confusao.png', bbox_inches='tight')
plt.show()

print(f"\nVN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  VP={cm[1,1]}")


## 10. Relatório de Classificação

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Rocha (0)", "Mina (1)"]))


## 11. Validação Cruzada (5-fold)

In [ ]:
X_scaled = StandardScaler().fit_transform(X)
cv_acc = cross_val_score(modelo_knn, X_scaled, y, cv=5, scoring='accuracy')
cv_f1  = cross_val_score(modelo_knn, X_scaled, y, cv=5, scoring='f1')

fig, ax = plt.subplots(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
ax.bar(x - 0.2, cv_acc, 0.35, label='Acurácia', color='#4CAF50', alpha=0.85)
ax.bar(x + 0.2, cv_f1,  0.35, label='F1-Score',  color='#FF9800', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(folds)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Validação Cruzada 5-Fold – KNN', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../images/knn_cv.png', bbox_inches='tight')
plt.show()

print(f"Acurácia CV → média={cv_acc.mean():.4f}  std={cv_acc.std():.4f}")
print(f"F1-Score CV → média={cv_f1.mean():.4f}   std={cv_f1.std():.4f}")


## 12. Sumário Final

In [ ]:
sumario = pd.DataFrame({
    "Modelo"                  : ["KNN (k=5)"],
    "Acurácia"                : [round(acc, 4)],
    "Precisão"                : [round(precision, 4)],
    "Recall"                  : [round(recall, 4)],
    "F1-Score"                : [round(f1, 4)],
    "ROC-AUC"                 : [round(roc_auc, 4)],
    "CV Acurácia (μ)"         : [round(cv_scores.mean(), 4)],
    "Tempo Treino (s)"        : [round(tempo_treino, 6)],
    "Tempo Inferência (s)"    : [round(tempo_inferencia, 6)],
    "Tempo/Amostra (ms)"      : [round(tempo_por_amostra, 4)],
    "Pico Memória Treino (MB)": [round(mem_pico_treino_mb, 4)],
    "Delta RAM (MB)"          : [round(ram_depois - ram_antes, 2)],
})
print("=== SUMÁRIO – KNN ===")
display(sumario)
print("\n✅ Salve este DataFrame para consolidar com os demais modelos.")
